# N2 — Convencionalización de metáforas (Procesamiento)
## AI-MELT: Nivel 2 — Metáfora convencional

**Propósito:** Determinar cuáles metáforas primarias se han estabilizado en la comunidad discursiva del Informe mediante el doble enfoque de MELT: frecuencia de aparición y agrupación conceptual por familias de dominio fuente.

**Entrada:** `n1_metaforas_primarias.csv` (salida de N1)

**Salida:**
- `n2_metaforas_convencionales.csv` — una fila por metáfora convencional
- `n2_primaria_convencional.csv` — tabla M:N que conecta primarias con convencionales
- `n2_embeddings_cache.npz` — embeddings precalculados (para visualización)
- `n2_metadata.json` — configuración y estadísticas de ejecución

**Visualización:** Ejecutar `N2_convencionalizacion_viz.ipynb` después de este notebook.

---

### Estructura
1. Configuración
2. Carga de datos N1
3. Enfoque A: Convencionalización por frecuencia
4. Enfoque B: Agrupación conceptual (clustering)
5. Medición de robustez
6. Consolidación y exportación

## 1. Configuración

In [ ]:
# ============================================================
# 1. DEPENDENCIAS E IMPORTS
# ============================================================
# Descomenta si necesitas instalar:
# !pip install sentence-transformers scikit-learn scipy umap-learn pandas tqdm

import json
import os
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')

print("✓ Imports completados")

In [ ]:
# ============================================================
# 1B. CONFIGURACIÓN
# ============================================================

# ┌─────────────────────────────────────────────────────────┐
# │  MODIFICA ESTA SECCIÓN SEGÚN TUS NECESIDADES            │
# └─────────────────────────────────────────────────────────┘

# Rutas
INPUT_DIR = "./outputs/N1/"
OUTPUT_DIR = "./outputs/N2/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Modelo de embeddings
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# --- Enfoque A: Frecuencia ---
# Umbral de similitud coseno para considerar dos formulaciones como "la misma metáfora"
SIMILARITY_THRESHOLD = 0.85

# Percentiles para clasificación de robustez
PERCENTIL_ALTA = 90      # frecuencia > percentil 90 → ALTA
PERCENTIL_MODERADA = 50  # frecuencia entre percentil 50 y 90 → MODERADA
                          # por debajo → DÉBIL

# --- Enfoque B: Clustering ---
# DBSCAN
DBSCAN_EPS = 0.35           # Radio máximo entre puntos del mismo cluster
DBSCAN_MIN_SAMPLES = 2      # Mínimo de puntos para formar un cluster

# K-means: rango para elbow method
KMEANS_K_RANGE = range(3, 25)

# Clustering jerárquico
HIERARCHICAL_METHOD = "ward"   # ward, complete, average, single
HIERARCHICAL_DISTANCE = 1.5    # Distancia de corte para fcluster

# --- Filtros ---
# Solo incluir enfoques de N1 que producen dominios completos
# (excluir standalone de HF/embeddings que tienen campos vacíos)
FILTER_EMPTY_DOMAINS = True

print("✓ Configuración cargada")
print(f"  Embedding model: {EMBEDDING_MODEL}")
print(f"  Similarity threshold: {SIMILARITY_THRESHOLD}")
print(f"  Clustering: DBSCAN(eps={DBSCAN_EPS}), K-means({KMEANS_K_RANGE.start}-{KMEANS_K_RANGE.stop-1}), Jerárquico({HIERARCHICAL_METHOD})")

## 2. Carga de datos N1

In [ ]:
# ============================================================
# 2. CARGA DE DATOS N1
# ============================================================

input_path = os.path.join(INPUT_DIR, "n1_metaforas_primarias.csv")
df_n1 = pd.read_csv(input_path)
print(f"✓ Cargado: {input_path}")
print(f"  Metáforas primarias: {len(df_n1):,}")
print(f"  Enfoques presentes: {df_n1['enfoque'].unique().tolist()}")

# Filtrar filas sin dominios si se configuró
if FILTER_EMPTY_DOMAINS:
    before = len(df_n1)
    df_n1 = df_n1[
        df_n1["dominio_fuente"].notna() & (df_n1["dominio_fuente"] != "") &
        df_n1["dominio_meta"].notna() & (df_n1["dominio_meta"] != "") &
        df_n1["metafora_conceptual"].notna() & (df_n1["metafora_conceptual"] != "")
    ].copy()
    print(f"  Filtradas por dominios completos: {before} → {len(df_n1)}")

# Normalizar formulaciones a mayúsculas para consistencia
df_n1["dominio_fuente_norm"] = df_n1["dominio_fuente"].str.upper().str.strip()
df_n1["dominio_meta_norm"] = df_n1["dominio_meta"].str.upper().str.strip()
df_n1["metafora_conceptual_norm"] = df_n1["metafora_conceptual"].str.upper().str.strip()

print(f"\n  Metáforas conceptuales únicas (texto): {df_n1['metafora_conceptual_norm'].nunique()}")
print(f"  Dominios fuente únicos: {df_n1['dominio_fuente_norm'].nunique()}")
print(f"  Dominios meta únicos: {df_n1['dominio_meta_norm'].nunique()}")

## 3. Enfoque A: Convencionalización por frecuencia

1. Genera embeddings de cada formulación de `metafora_conceptual`
2. Calcula similitud coseno entre todas las formulaciones
3. Agrupa formulaciones con similitud > umbral como "misma metáfora"
4. Cuenta frecuencia de cada grupo
5. Clasifica robustez por percentiles

In [ ]:
# ============================================================
# 3A. EMBEDDINGS DE METÁFORAS CONCEPTUALES
# ============================================================

print(f"Cargando modelo de embeddings: {EMBEDDING_MODEL}...")
model_emb = SentenceTransformer(EMBEDDING_MODEL)

# Formulaciones únicas
unique_mc = df_n1["metafora_conceptual_norm"].unique().tolist()
print(f"Formulaciones únicas a procesar: {len(unique_mc)}")

# Generar embeddings
print("Generando embeddings...")
mc_embeddings = model_emb.encode(unique_mc, show_progress_bar=True, batch_size=64,
                                  convert_to_numpy=True, normalize_embeddings=True)

# Mapa: formulación → índice
mc_to_idx = {mc: i for i, mc in enumerate(unique_mc)}

print(f"✓ Embeddings generados: shape={mc_embeddings.shape}")

In [ ]:
# ============================================================
# 3B. AGRUPACIÓN POR SIMILITUD (ENFOQUE FRECUENCIA)
# ============================================================

# Matriz de similitud coseno (ya están normalizados → dot product)
print("Calculando matriz de similitud...")
sim_matrix = mc_embeddings @ mc_embeddings.T

# Agrupación greedy: para cada formulación, buscar las que superan el umbral
# y asignarlas al mismo grupo
visited = set()
frequency_groups = []  # Lista de sets, cada set = un grupo de formulaciones equivalentes

for i, mc in enumerate(unique_mc):
    if i in visited:
        continue
    group = {i}
    for j in range(i + 1, len(unique_mc)):
        if j not in visited and sim_matrix[i, j] >= SIMILARITY_THRESHOLD:
            group.add(j)
    visited.update(group)
    frequency_groups.append(group)

print(f"✓ Grupos de frecuencia formados: {len(frequency_groups)}")
print(f"  Formulaciones originales: {len(unique_mc)}")
print(f"  Reducción: {len(unique_mc)} → {len(frequency_groups)} ({(1 - len(frequency_groups)/len(unique_mc))*100:.1f}%)")

# Para cada grupo, elegir la formulación más frecuente como representante
freq_results = []
for group_idx, group in enumerate(frequency_groups):
    group_formulations = [unique_mc[i] for i in group]

    # Contar instancias totales en df_n1 para este grupo
    mask = df_n1["metafora_conceptual_norm"].isin(group_formulations)
    n_instances = mask.sum()

    # Formulación más frecuente como representante
    freq_by_form = df_n1[mask]["metafora_conceptual_norm"].value_counts()
    representative = freq_by_form.index[0]

    # Dominio fuente y meta más frecuentes del grupo
    dom_fuente = df_n1[mask]["dominio_fuente_norm"].value_counts().index[0]
    dom_meta = df_n1[mask]["dominio_meta_norm"].value_counts().index[0]

    freq_results.append({
        "group_id": group_idx,
        "metafora_conceptual": representative,
        "dominio_fuente": dom_fuente,
        "dominio_meta": dom_meta,
        "frecuencia": int(n_instances),
        "n_formulaciones_agrupadas": len(group),
        "formulaciones_agrupadas": json.dumps(group_formulations, ensure_ascii=False),
        "enfoque_origen": "frecuencia"
    })

df_freq = pd.DataFrame(freq_results).sort_values("frecuencia", ascending=False).reset_index(drop=True)

# Clasificar robustez por percentiles
p_alta = np.percentile(df_freq["frecuencia"], PERCENTIL_ALTA)
p_mod = np.percentile(df_freq["frecuencia"], PERCENTIL_MODERADA)

def classify_robustez(freq):
    if freq >= p_alta:
        return "ALTA"
    elif freq >= p_mod:
        return "MODERADA"
    return "DÉBIL"

df_freq["robustez_frecuencia"] = df_freq["frecuencia"].apply(classify_robustez)

print("\nDistribución de robustez (enfoque frecuencia):")
print(df_freq["robustez_frecuencia"].value_counts().to_string())
print("\nTop 10 metáforas convencionales por frecuencia:")
print(df_freq[["metafora_conceptual", "frecuencia", "robustez_frecuencia"]].head(10).to_string(index=False))

## 4. Enfoque B: Agrupación conceptual (clustering)

Agrupa las metáforas primarias por similitud de `dominio_fuente` usando tres algoritmos para comparar.

In [ ]:
# ============================================================
# 4A. EMBEDDINGS DE DOMINIOS FUENTE
# ============================================================

unique_df = df_n1["dominio_fuente_norm"].unique().tolist()
print(f"Dominios fuente únicos: {len(unique_df)}")

df_embeddings = model_emb.encode(unique_df, show_progress_bar=True, batch_size=64,
                                  convert_to_numpy=True, normalize_embeddings=True)
df_to_idx = {df_name: i for i, df_name in enumerate(unique_df)}

print(f"✓ Embeddings de dominios fuente: shape={df_embeddings.shape}")

In [ ]:
# ============================================================
# 4B. DBSCAN
# ============================================================

print(f"DBSCAN (eps={DBSCAN_EPS}, min_samples={DBSCAN_MIN_SAMPLES})...")
dbscan = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES, metric="cosine")  # DBSCAN es determinista por diseño
labels_dbscan = dbscan.fit_predict(df_embeddings)

n_clusters_dbscan = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
n_outliers_dbscan = (labels_dbscan == -1).sum()

sil_dbscan = None
if n_clusters_dbscan >= 2 and n_outliers_dbscan < len(labels_dbscan):
    mask_valid = labels_dbscan != -1
    if mask_valid.sum() >= 2:
        sil_dbscan = silhouette_score(df_embeddings[mask_valid], labels_dbscan[mask_valid], metric="cosine")

print(f"  Clusters: {n_clusters_dbscan}")
print(f"  Outliers: {n_outliers_dbscan}")
print(f"  Silhouette: {sil_dbscan:.3f}" if sil_dbscan else "  Silhouette: N/A")

In [ ]:
# ============================================================
# 4C. K-MEANS CON ELBOW METHOD
# ============================================================

print(f"K-means: evaluando k={KMEANS_K_RANGE.start} a {KMEANS_K_RANGE.stop-1}...")
inertias = []
silhouettes_km = []
valid_ks = []

for k in KMEANS_K_RANGE:
    if k >= len(df_embeddings):
        break
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels_km_temp = km.fit_predict(df_embeddings)
    inertias.append(km.inertia_)
    sil = silhouette_score(df_embeddings, labels_km_temp, metric="cosine")
    silhouettes_km.append(sil)
    valid_ks.append(k)

# Seleccionar k con mejor silhouette
best_k_idx = np.argmax(silhouettes_km)
best_k = valid_ks[best_k_idx]
best_sil_km = silhouettes_km[best_k_idx]

print(f"  Mejor k: {best_k} (silhouette={best_sil_km:.3f})")

# Ajustar con el mejor k
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels_kmeans = km_final.fit_predict(df_embeddings)

print(f"  Clusters finales: {best_k}")

In [ ]:
# ============================================================
# 4D. CLUSTERING JERÁRQUICO
# ============================================================

print(f"Clustering jerárquico (método={HIERARCHICAL_METHOD}, distancia={HIERARCHICAL_DISTANCE})...")

# Linkage matrix para dendrograma (se guarda para visualización)
dist_matrix = 1 - (df_embeddings @ df_embeddings.T)
np.fill_diagonal(dist_matrix, 0)
dist_matrix = np.clip(dist_matrix, 0, None)  # Asegurar no negativos
condensed_dist = squareform(dist_matrix, checks=False)
linkage_matrix = linkage(condensed_dist, method=HIERARCHICAL_METHOD)

# Cortar en clusters
labels_hierarchical = fcluster(linkage_matrix, t=HIERARCHICAL_DISTANCE, criterion="distance") - 1

n_clusters_hier = len(set(labels_hierarchical))
sil_hier = silhouette_score(df_embeddings, labels_hierarchical, metric="cosine") if n_clusters_hier >= 2 else None

print(f"  Clusters: {n_clusters_hier}")
print(f"  Silhouette: {sil_hier:.3f}" if sil_hier else "  Silhouette: N/A")

In [ ]:
# ============================================================
# 4E. COMPARACIÓN DE ALGORITMOS Y SELECCIÓN
# ============================================================

comparison = {
    "DBSCAN": {"n_clusters": n_clusters_dbscan, "silhouette": sil_dbscan, "labels": labels_dbscan},
    "K-means": {"n_clusters": best_k, "silhouette": best_sil_km, "labels": labels_kmeans},
    "Jerárquico": {"n_clusters": n_clusters_hier, "silhouette": sil_hier, "labels": labels_hierarchical},
}

print("COMPARACIÓN DE ALGORITMOS DE CLUSTERING")
print("=" * 55)
print(f"{'Algoritmo':<15} {'Clusters':>10} {'Silhouette':>12}")
print("-" * 55)
best_algo = None
best_sil = -1
for algo, info in comparison.items():
    sil_str = f"{info['silhouette']:.3f}" if info['silhouette'] is not None else "N/A"
    print(f"{algo:<15} {info['n_clusters']:>10} {sil_str:>12}")
    if info['silhouette'] is not None and info['silhouette'] > best_sil:
        best_sil = info['silhouette']
        best_algo = algo

print(f"\n→ Mejor algoritmo por silhouette: {best_algo} ({best_sil:.3f})")

# Usar el mejor para construir los clusters conceptuales
selected_labels = comparison[best_algo]["labels"]

# Construir clusters de dominios fuente
cluster_groups = defaultdict(list)
for dom_fuente, label in zip(unique_df, selected_labels, strict=False):
    if label != -1:  # Excluir outliers de DBSCAN
        cluster_groups[int(label)].append(dom_fuente)

print(f"\nClusters conceptuales ({best_algo}):")
for cid in sorted(cluster_groups.keys()):
    members = cluster_groups[cid]
    print(f"  Cluster {cid}: {len(members)} dominios — {', '.join(members[:5])}{'...' if len(members) > 5 else ''}")

In [ ]:
# ============================================================
# 4F. METÁFORAS CONVENCIONALES POR CLUSTERING
# ============================================================

# Para cada cluster, encontrar la metáfora conceptual más frecuente
# entre las metáforas primarias cuyos dominios fuente pertenecen al cluster

cluster_results = []
for cid, dom_fuentes in cluster_groups.items():
    mask = df_n1["dominio_fuente_norm"].isin(dom_fuentes)
    df_cluster = df_n1[mask]

    if len(df_cluster) == 0:
        continue

    # Metáfora conceptual más frecuente del cluster = la estructurante
    mc_freq = df_cluster["metafora_conceptual_norm"].value_counts()
    representative = mc_freq.index[0]
    dom_fuente_principal = df_cluster["dominio_fuente_norm"].value_counts().index[0]
    dom_meta_principal = df_cluster["dominio_meta_norm"].value_counts().index[0]

    cluster_results.append({
        "cluster_id": cid,
        "metafora_conceptual": representative,
        "dominio_fuente": dom_fuente_principal,
        "dominio_meta": dom_meta_principal,
        "frecuencia": int(len(df_cluster)),
        "n_dominios_fuente": len(dom_fuentes),
        "dominios_fuente_agrupados": json.dumps(dom_fuentes, ensure_ascii=False),
        "enfoque_origen": "clustering",
        "algoritmo_clustering": best_algo,
    })

df_cluster_conv = pd.DataFrame(cluster_results).sort_values("frecuencia", ascending=False).reset_index(drop=True)
print(f"✓ Metáforas convencionales por clustering: {len(df_cluster_conv)}")

## 5. Medición de robustez

La robustez mide cuán extendida está una metáfora convencional a lo largo del corpus:
- **Dispersión por documentos:** ¿aparece en un solo capítulo o en todo el informe?
- **Coeficiente de variación:** ¿su frecuencia es uniforme o concentrada?

In [ ]:
# ============================================================
# 5. MEDICIÓN DE ROBUSTEZ
# ============================================================

def compute_robustness(df_source, mc_column, doc_column="ID_documento"):
    """Calcula métricas de dispersión textual para cada metáfora convencional."""
    results = {}
    all_docs = df_source[doc_column].nunique()

    for mc in df_source[mc_column].unique():
        mask = df_source[mc_column] == mc
        docs_present = df_source[mask][doc_column].nunique()
        freq_by_doc = df_source[mask].groupby(doc_column).size()

        dispersion = docs_present / all_docs if all_docs > 0 else 0
        cv = freq_by_doc.std() / freq_by_doc.mean() if freq_by_doc.mean() > 0 else 0

        # Distribución textual (para N3)
        dist_textual = {}
        for _, row in df_source[mask].iterrows():
            dom_f = row.get("dominio_fuente_norm", "")
            key = dom_f
            if key not in dist_textual:
                dist_textual[key] = {
                    "dominio_meta": row.get("dominio_meta_norm", ""),
                    "paginas": [],
                    "expresiones": [],
                }
            dist_textual[key]["paginas"].append(row.get("pagina", ""))
            expr = row.get("expresion_metaforica", "")
            if expr and expr not in dist_textual[key]["expresiones"]:
                dist_textual[key]["expresiones"].append(expr)

        results[mc] = {
            "n_documentos": int(docs_present),
            "total_documentos": int(all_docs),
            "dispersion": round(float(dispersion), 3),
            "coef_variacion": round(float(cv), 3),
            "distribucion_textual": json.dumps(dist_textual, ensure_ascii=False)
        }

    return results

# Calcular robustez para el enfoque de frecuencia
print("Calculando robustez para enfoque de frecuencia...")
robustez_freq = {}
for _, row in df_freq.iterrows():
    formulations = json.loads(row["formulaciones_agrupadas"])
    mask = df_n1["metafora_conceptual_norm"].isin(formulations)
    df_sub = df_n1[mask]
    all_docs = df_n1["ID_documento"].nunique()
    docs_present = df_sub["ID_documento"].nunique()
    freq_by_doc = df_sub.groupby("ID_documento").size()
    dispersion = docs_present / all_docs if all_docs > 0 else 0
    cv = freq_by_doc.std() / freq_by_doc.mean() if freq_by_doc.mean() > 0 else 0
    robustez_freq[row["metafora_conceptual"]] = {
        "dispersion": round(float(dispersion), 3),
        "coef_variacion": round(float(cv), 3),
        "n_documentos": int(docs_present)
    }

df_freq["dispersion"] = df_freq["metafora_conceptual"].map(lambda x: robustez_freq.get(x, {}).get("dispersion", 0))
df_freq["coef_variacion"] = df_freq["metafora_conceptual"].map(lambda x: robustez_freq.get(x, {}).get("coef_variacion", 0))
df_freq["n_documentos"] = df_freq["metafora_conceptual"].map(lambda x: robustez_freq.get(x, {}).get("n_documentos", 0))

# Clasificar robustez combinada (frecuencia + dispersión)
def robustez_combinada(row):
    if row["robustez_frecuencia"] == "ALTA" and row["dispersion"] >= 0.5:
        return "ALTA"
    elif row["robustez_frecuencia"] in ["ALTA", "MODERADA"] and row["dispersion"] >= 0.3:
        return "MODERADA"
    return "DÉBIL"

df_freq["robustez"] = df_freq.apply(robustez_combinada, axis=1)

# Calcular distribución textual para todos
robustez_all = compute_robustness(df_n1, "metafora_conceptual_norm")

print("✓ Robustez calculada")
print("\nDistribución de robustez combinada:")
print(df_freq["robustez"].value_counts().to_string())

## 6. Consolidación y exportación

In [ ]:
# ============================================================
# 6A. CONSTRUIR CSV DE METÁFORAS CONVENCIONALES
# ============================================================

rows_consolidated = []
id_counter = 0

# Añadir resultados del enfoque de frecuencia
for _, row in df_freq.iterrows():
    id_counter += 1
    mc_key = row["metafora_conceptual"]
    rows_consolidated.append({
        "ID_metaconvencional": f"MC-FRQ-{id_counter:04d}",
        "metafora_conceptual": row["metafora_conceptual"],
        "dominio_fuente": row["dominio_fuente"],
        "dominio_meta": row["dominio_meta"],
        "enfoque_origen": "frecuencia",
        "frecuencia_absoluta": row["frecuencia"],
        "n_formulaciones_agrupadas": row["n_formulaciones_agrupadas"],
        "cluster_id": "",
        "algoritmo_clustering": "",
        "robustez": row["robustez"],
        "dispersion": row["dispersion"],
        "coef_variacion": row["coef_variacion"],
        "n_documentos": row["n_documentos"],
        "distribucion_textual": robustez_all.get(mc_key, {}).get("distribucion_textual", "{}"),
    })

# Añadir resultados del enfoque de clustering
for _, row in df_cluster_conv.iterrows():
    id_counter += 1
    mc_key = row["metafora_conceptual"]
    rob = robustez_all.get(mc_key, {})
    rows_consolidated.append({
        "ID_metaconvencional": f"MC-CLU-{id_counter:04d}",
        "metafora_conceptual": row["metafora_conceptual"],
        "dominio_fuente": row["dominio_fuente"],
        "dominio_meta": row["dominio_meta"],
        "enfoque_origen": "clustering",
        "frecuencia_absoluta": row["frecuencia"],
        "n_formulaciones_agrupadas": row.get("n_dominios_fuente", 1),
        "cluster_id": row["cluster_id"],
        "algoritmo_clustering": row["algoritmo_clustering"],
        "robustez": "MODERADA",  # Por defecto; se refina con métricas
        "dispersion": rob.get("dispersion", 0),
        "coef_variacion": rob.get("coef_variacion", 0),
        "n_documentos": rob.get("n_documentos", 0),
        "distribucion_textual": rob.get("distribucion_textual", "{}"),
    })

df_convencionales = pd.DataFrame(rows_consolidated)

# Exportar
path_conv = os.path.join(OUTPUT_DIR, "n2_metaforas_convencionales.csv")
df_convencionales.to_csv(path_conv, index=False, encoding='utf-8-sig')
print(f"✓ {path_conv} ({len(df_convencionales):,} filas)")

In [ ]:
# ============================================================
# 6B. TABLA M:N PRIMARIA ↔ CONVENCIONAL
# ============================================================

mn_rows = []

# Para el enfoque de frecuencia: conectar cada primaria con su grupo
for _, row in df_freq.iterrows():
    formulations = json.loads(row["formulaciones_agrupadas"])
    id_mc = f"MC-FRQ-{row['group_id'] + 1:04d}"
    matching = df_n1[df_n1["metafora_conceptual_norm"].isin(formulations)]
    for _, prim in matching.iterrows():
        mn_rows.append({
            "ID_metaconvencional": id_mc,
            "ID_expresion": prim["ID_expresion"]
        })

# Para el enfoque de clustering
for _, row in df_cluster_conv.iterrows():
    doms = json.loads(row["dominios_fuente_agrupados"])
    id_mc = f"MC-CLU-{row.name + len(df_freq) + 1:04d}"
    matching = df_n1[df_n1["dominio_fuente_norm"].isin(doms)]
    for _, prim in matching.iterrows():
        mn_rows.append({
            "ID_metaconvencional": id_mc,
            "ID_expresion": prim["ID_expresion"]
        })

df_mn = pd.DataFrame(mn_rows).drop_duplicates()
path_mn = os.path.join(OUTPUT_DIR, "n2_primaria_convencional.csv")
df_mn.to_csv(path_mn, index=False, encoding='utf-8-sig')
print(f"✓ {path_mn} ({len(df_mn):,} relaciones)")

In [ ]:
# ============================================================
# 6C. GUARDAR EMBEDDINGS Y ARTEFACTOS PARA VISUALIZACIÓN
# ============================================================

# Guardar embeddings y labels para el notebook de visualización
np.savez_compressed(
    os.path.join(OUTPUT_DIR, "n2_embeddings_cache.npz"),
    mc_embeddings=mc_embeddings,
    df_embeddings=df_embeddings,
    sim_matrix=sim_matrix,
    linkage_matrix=linkage_matrix,
)

# Guardar listas de labels y nombres
artifacts = {
    "unique_mc": unique_mc,
    "unique_df": unique_df,
    "labels_dbscan": labels_dbscan.tolist(),
    "labels_kmeans": labels_kmeans.tolist(),
    "labels_hierarchical": labels_hierarchical.tolist(),
    "best_algo": best_algo,
    "inertias": inertias,
    "silhouettes_km": silhouettes_km,
    "valid_ks": [int(k) for k in valid_ks],
    "best_k": int(best_k),
    "comparison": {algo: {"n_clusters": info["n_clusters"],
                           "silhouette": float(info["silhouette"]) if info["silhouette"] else None}
                    for algo, info in comparison.items()},
}
with open(os.path.join(OUTPUT_DIR, "n2_artifacts.json"), 'w', encoding='utf-8') as f:
    json.dump(artifacts, f, ensure_ascii=False, indent=2)

print("✓ n2_embeddings_cache.npz")
print("✓ n2_artifacts.json")

## 6E. Muestra para evaluación human-in-the-loop

Muestra aleatoria de metáforas convencionales para que el analista verifique:
- ¿Las formulaciones agrupadas en un mismo grupo son realmente la misma metáfora?
- ¿Los clusters conceptuales agrupan dominios fuente coherentes?

In [ ]:
# ============================================================
# 6E. MUESTRA HUMAN-IN-THE-LOOP
# ============================================================

EVAL_SAMPLE = min(30, len(df_convencionales))

if EVAL_SAMPLE > 0:
    df_eval = df_convencionales.sample(n=EVAL_SAMPLE, random_state=42)[
        ["ID_metaconvencional", "metafora_conceptual", "dominio_fuente", "dominio_meta",
         "enfoque_origen", "frecuencia_absoluta", "robustez"]
    ].copy()

    # Columnas para evaluación manual
    df_eval["agrupacion_correcta"] = ""       # Sí / No / Parcial
    df_eval["robustez_correcta"] = ""         # Sí / No
    df_eval["observaciones"] = ""

    eval_path = os.path.join(OUTPUT_DIR, "n2_muestra_evaluacion.csv")
    df_eval.to_csv(eval_path, index=False, encoding='utf-8-sig')
    print(f"✓ {eval_path} ({EVAL_SAMPLE} metáforas convencionales)")
    print("  Columnas para evaluar: agrupacion_correcta, robustez_correcta, observaciones")
else:
    print("⚠ Sin datos para muestra de evaluación.")

In [ ]:
# ============================================================
# 6D. METADATOS Y RESUMEN
# ============================================================

metadata = {
    "nivel": "N2",
    "embedding_model": EMBEDDING_MODEL,
    "similarity_threshold": SIMILARITY_THRESHOLD,
    "metaforas_primarias_entrada": len(df_n1),
    "metaforas_convencionales_frecuencia": len(df_freq),
    "metaforas_convencionales_clustering": len(df_cluster_conv),
    "total_metaforas_convencionales": len(df_convencionales),
    "relaciones_mn": len(df_mn),
    "mejor_algoritmo_clustering": best_algo,
    "clustering_comparison": artifacts["comparison"],
}
with open(os.path.join(OUTPUT_DIR, "n2_metadata.json"), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("=" * 60)
print("N2 — CONVENCIONALIZACIÓN COMPLETADO")
print("=" * 60)
print(f"\nMetáforas convencionales por frecuencia: {len(df_freq)}")
print(f"Metáforas convencionales por clustering: {len(df_cluster_conv)}")
print(f"Total: {len(df_convencionales)}")
print(f"Relaciones M:N: {len(df_mn):,}")
print(f"\nArchivos en {OUTPUT_DIR}:")
for f_name in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f_name)) / 1024
    print(f"  📄 {f_name} ({size:.0f} KB)")
print("\n➡ SIGUIENTE: Ejecutar N2_convencionalizacion_viz.ipynb para visualizaciones")
print("➡ LUEGO: Ejecutar N3_escenarios.ipynb")